# Phase 2: GeoJSON streaming export

This notebook implements the Phase 2 processing pipeline. Here, we add synthetic latitude and longitude to the telemetry stream, which allows our simulated agents to have coordinate trajectories. The stream is synchronously processed via Spark Structured Streaming, exporting each micro-batch into a `GeoJSON` `FeatureCollection` format for geospatial analysis.

*Note*: The coordinates are simulated programmatically for pipeline demonstration purposes and do not represent actual GPS observations.


In [3]:
import os
import sys
import logging
from pathlib import Path
from pyspark.sql import SparkSession
import sys; sys.path.append(os.path.abspath('..'))
from src.bootstrapping import setup_winutils
# Climb up from the notebook's folder to find the true project workspace root
notebook_dir = Path(os.getcwd())
PROJECT_ROOT = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# Configure Windows-specific local Spark settings dynamically
if os.name == 'nt':
    setup_winutils(PROJECT_ROOT)
    os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
# Check if an active Spark session already exists or is configured in the environment.
active_session = SparkSession.getActiveSession()
if active_session is not None:
    spark = active_session
    logging.info("Reusing active Spark Session.")
else:
    logging.info("Spawning adaptive geospatial Spark Session environment...")
    spark_builder = (
        SparkSession.builder
        .appName('Geospatial-Streaming-Export')
        .config('spark.sql.shuffle.partitions', '4')
    )
    if os.name == 'nt':
        spark_builder = (
            spark_builder
            .config('spark.driver.host', '127.0.0.1')
            .config('spark.pyspark.python', sys.executable)
            .config('spark.pyspark.driver.python', sys.executable)
        )
    master_url = os.environ.get("SPARK_MASTER")
    if not master_url and not any(env.startswith("SPARK_") for env in os.environ):
        spark_builder = spark_builder.master("local[*]") \
                                     .config("spark.driver.memory", "4g")
    spark = spark_builder.getOrCreate()
logging.info(f'Project root: {PROJECT_ROOT}')


2026-07-08 15:05:01,241 - INFO - Hadoop environment path configuration active: HADOOP_HOME=c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils
2026-07-08 15:05:01,250 - INFO - Spawning adaptive geospatial Spark Session environment...


KeyboardInterrupt: 

## 2.1 Spatial trajectory generation and coordinate projection

As our sensor data time-series has a dimension of 1, for a 2D geographical space the pipeline attaches the synthetically generated latitude ($\phi$) and longitude ($\lambda$) to each agent.

#### 1. Velocity Models
An agent's speed $v$ is determined by its classified activity state:
$$v = \begin{cases} 1.4\text{ m/s} & \text{if Activity} = \text{"walk"} \\ 4.5\text{ m/s} & \text{if Activity} = \text{"bike"} \\ 0.0\text{ m/s} & \text{otherwise} \end{cases}$$

#### 2. Angular Heading Direction
The walking or cycling heading direction $\theta$ (in radians) is determined by the agent's identifier modulo 16, dividing the compass into 16 discrete angles:
$$\theta = (\text{Agent\_ID} \pmod{16}) \times \frac{2\pi}{16}$$

#### 3. Spatial Displacement Mapping
Given the elapsed time $t$ since the base timestamp, the displacement in metres along the East ($\Delta x$) and North ($\Delta y$) axes is calculated as:
$$\Delta x = v \cdot t \cdot \cos(\theta)$$
$$\Delta y = v \cdot t \cdot \sin(\theta)$$

Using a local projection centred on Vienna ($\phi_{\text{start}} = 48.20849^\circ\text{ N}$, $\lambda_{\text{start}} = 16.37208^\circ\text{ E}$), we convert these displacements to degree coordinates:
$$\phi = \phi_{\text{start}} + \frac{\Delta y}{M_{\text{lat}}}$$
$$\lambda = \lambda_{\text{start}} + \frac{\Delta x}{M_{\text{lon}}}$$
where the metres-to-degrees conversion factors are defined by:
$$M_{\text{lat}} = 111,320.0 \text{ m/degree}$$
$$M_{\text{lon}} = 111,320.0 \cdot \cos(\phi_{\text{start}}) \text{ m/degree}$$


> [!NOTE]
> **Memory efficiency justification**
> The following simulation triggers `run_geospatial_streaming_simulation`, which downloads Vienna's spatial layers, snaps agent starting points, and resolves routes via `OSMnx`. It collects the distinct list of agents (limited to 150 synthetic subjects) to the driver to calculate spatial path coordinates. Collecting this small metadata list is computationally safe and does not cause driver memory overflow. The streaming coordinates are then iterated using `toLocalIterator()` to write the GeoJSON feature batches page-by-page, preventing out-of-memory errors.


In [ ]:
from src.geospatial_streaming import run_geospatial_streaming_simulation
output_dir = run_geospatial_streaming_simulation(spark, PROJECT_ROOT)
geojson_files = sorted(output_dir.glob('*.geojson'))
print(f'Created {len(geojson_files)} GeoJSON micro-batch files in {output_dir}')
geojson_files[:3]


2026-07-08 03:19:53,410 - INFO - Layer already cached: vienna_districts.geojson
2026-07-08 03:19:53,418 - INFO - Layer already cached: vienna_pedestrian_zones.geojson
2026-07-08 03:19:53,426 - INFO - Layer already cached: vienna_bike_paths.geojson
2026-07-08 03:20:00,677 - INFO - Loading cached street graph: vienna_walk_network.graphml
2026-07-08 03:24:36,922 - INFO - Geospatial streaming export completed. Output: c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output


Created 10 GeoJSON micro-batch files in c:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\geospatial_output


[WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00000.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00001.geojson'),
 WindowsPath('c:/Users/fedka/Documents/GitHub/Geospatial Repletion & Saturation Modelling/data/geospatial_output/telemetry_batch_00002.geojson')]

In [ ]:
import json
with geojson_files[0].open('r', encoding='utf-8') as file_handle:
    first_batch = json.load(file_handle)
print(first_batch['type'])
print(f"Features in first batch: {len(first_batch['features'])}")
first_batch['features'][0]


FeatureCollection
Features in first batch: 149997


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [16.3742637, 48.1840581]},
 'properties': {'Agent_ID': 0,
  'Timestamp': 1700000000130,
  'Activity': 'walk',
  'ax': 4.9671001751698975,
  'ay': 2.349444609099316,
  'az': 14.537952461505508,
  'gx': -1.3137137938307457,
  'gy': 0.5762100371226986,
  'gz': 0.3528859605986052,
  'coordinate_source': 'synthetic'}}

## 2.2 Interpretation and limitations

The generated GeoJSON files demonstrate how a Spark Structured Streaming pipeline can attach a geographical representation to synthetic telemetry. The coordinates are generated from explicit speed and direction assumptions; they are not observed GPS positions and are not snapped to official Vienna transport infrastructure. Consequently, the output is suitable for demonstrating data processing and visualisation, but not for drawing conclusions about actual Vienna mobility or congestion.


In [ ]:
try:
    logging.info("Shutting down Spark Session...")
finally:
    spark.stop()
    logging.info("Spark Session terminated successfully.")
